# Phase noise: Free-run и SIL

In [ ]:
import colorsys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


def hsl_to_rgb_normalized(h, s, l):
    return colorsys.hls_to_rgb(h / 360, l / 100, s / 100)


BLUE_RGB = hsl_to_rgb_normalized(206, 73, 48)
RED_RGB = hsl_to_rgb_normalized(9, 98, 63)

plt.rcParams.update(
    {
        "font.size": 24,
        "axes.titlesize": 24,
        "axes.labelsize": 24,
        "xtick.labelsize": 24,
        "ytick.labelsize": 24,
        "legend.fontsize": 24,
    }
)

In [ ]:
def load_phase_noise_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        sep=",",
        skiprows=3,
        header=None,
        names=["frequency_hz", "phase_noise_dbc_hz", "phase_noise_corrected_dbc_hz"],
        dtype=str,
        skip_blank_lines=True,
    )

    for col in ["frequency_hz", "phase_noise_dbc_hz", "phase_noise_corrected_dbc_hz"]:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(",", ".", regex=False)
            .replace({"--": pd.NA, "": pd.NA})
        )

    df["frequency_hz"] = pd.to_numeric(df["frequency_hz"], errors="raise")
    df["frequency_khz"] = df["frequency_hz"] / 1000.0

    df["phase_noise_dbc_hz"] = pd.to_numeric(df["phase_noise_dbc_hz"], errors="raise")
    df["phase_noise_corrected_dbc_hz"] = pd.to_numeric(
        df["phase_noise_corrected_dbc_hz"], errors="raise"
    )
    return df.sort_values("frequency_hz").reset_index(drop=True)


DATA_DIR = Path("..") / "data" / "Характеристики излучения"
phase_fr = load_phase_noise_csv(DATA_DIR / "Phase_noise_FR.csv")
phase_sil = load_phase_noise_csv(DATA_DIR / "Phase_noise_SIL.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(
    phase_sil["frequency_khz"],
    phase_sil["phase_noise_corrected_dbc_hz"],
    color=RED_RGB,
    linewidth=1.0,
    label="SIL",
)
ax.plot(
    phase_fr["frequency_khz"],
    phase_fr["phase_noise_corrected_dbc_hz"],
    color=BLUE_RGB,
    linewidth=1.0,
    label="Free-run",
)

ax.set_xlabel("Frequency, kHz")
ax.set_ylabel("Phase noise, dBc/Hz")

ax.set_xlim(0, 100)
ax.set_ylim(-140, -10)

ax.grid(False)

ax.legend(loc="upper right", frameon=True)

plt.tight_layout()
plt.show()